# Match Simbad Targets with LSSTCam Sources Catalogue

For each target star (identified by `simbad_id`, `ra_deg`, `dec_deg`) read from
the input CSV, this notebook:

1. Locates the Butler tract/patch that covers the target coordinates.
2. Loads the corresponding `source` (or equivalent catalogue dataset
   auto-discovered from the registry).
3. Performs a nearest-neighbour sky cross-match using
   `astropy.coordinates.SkyCoord.match_to_catalog_sky`.
4. Retains matches within a configurable search radius.
5. Saves the merged table (targets + matched LSST object columns) to CSV/Parquet.


- refer to : https://github.com/lsst/tutorial-notebooks/blob/main/DP1/200_Data_Products/201_Catalogs/201_2_Source_table.ipynb

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-27
- **Last update:** 2026-06-27


## Imports

In [ ]:
import gc
import logging
import os
import re
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time

from lsst.daf.butler import Butler, Timespan
import lsst.geom as geom
from lsst.geom import SpherePoint, degrees

In [ ]:
from libExtractLightcurves import safe_name, find_col, dataset_type_exists
from libExtractLightcurves import probe_schema
from libExtractLightcurves import save_per_star

## Usefull functions

### Main Extraction Loop function

In [ ]:
def extract_lightcurves(
    butler: Butler,
    df_targets: pd.DataFrame,
    timespan: Timespan,
    ra_col: str,
    dec_col: str,
    id_col: str,
    src_columns_avail: list[str],
    match_radius_arcsec: float,
    out_csv: str,
    out_summary_csv: str,
) -> None:
    """Iterate over every target and every (visit, detector) ref.

    Performs the sky cross-match and writes matched rows to *out_csv*
    incrementally (append mode after each star) to keep RAM usage constant.

    Memory strategy
    ---------------
    - Source tables are loaded one at a time and released after the match.
    - No cache is kept in memory.
    - Matched rows are flushed to CSV after each star so the in-memory
      accumulator never grows beyond one star's worth of data.
    """
    # Columns already stored as dedicated metadata fields — skip when copying
    # photometric columns to avoid duplication.
    skip_in_row = {
        ra_col,
        dec_col,
        id_col,
        "visit",
        "detector",
        "band",
        "day_obs",
        "physical_filter",
    }
    photo_cols = [c for c in src_columns_avail if c not in skip_in_row]

    # Write CSV header (empty DataFrame with the right columns).
    header_row: dict = {
        "simbad_id": "",
        "target_ra": 0.0,
        "target_dec": 0.0,
        "visit": 0,
        "detector": 0,
        "band": "",
        "day_obs": 0,
        "physical_filter": "",
        "sep_arcsec": 0.0,
        "src_ra": 0.0,
        "src_dec": 0.0,
        "sourceId": 0,
    }
    for c in photo_cols:
        header_row[c] = 0.0
    pd.DataFrame([header_row]).head(0).to_csv(out_csv, index=False)

    summary_rows: list[dict] = []

    # loop on simbad targets
    for idx, target in df_targets.iterrows():
        simbad_id = target["simbad_id"]
        ra_t = float(target["ra_deg"])
        dec_t = float(target["dec_deg"])
        field = target["field"]

        # target coordinates
        tgt_sky = SkyCoord(ra=ra_t * u.deg, dec=dec_t * u.deg)

        log.info("[%3d] %s (%s) ra=%.5f  dec=%+.5f", idx, simbad_id, field, ra_t, dec_t)

        # 1. Query refs
        try:
            refs = list(
                butler.query_datasets(
                    "source",
                    where=(
                        "visit.timespan OVERLAPS :timespan AND "
                        "visit_detector_region.region OVERLAPS POINT(:ra, :dec)"
                    ),
                    bind={"timespan": timespan, "ra": ra_t, "dec": dec_t},
                )
            )
        except Exception as exc:
            log.error("  ERROR querying refs: %s", exc)
            summary_rows.append(
                {
                    "simbad_id": simbad_id,
                    "ra_deg": ra_t,
                    "dec_deg": dec_t,
                    "n_refs": 0,
                    "n_matched": 0,
                    "n_failed": 0,
                    "status": "query_error",
                }
            )
            continue

        log.info("  -> %d refs (visit x detector pairs)", len(refs))

        n_matched = 0
        n_failed = 0
        batch: list[dict] = []

        for count, ref in enumerate(refs):
            did = ref.dataId
            visit = did["visit"]
            band = did.get("band", "?")
            day_obs = did.get("day_obs", -1)
            detector = did.get("detector", -1)
            physical_filter = did.get("physical_filter", "?")

            if count % 50 == 0:
                log.info(
                    "    ref %d/%d  visit=%s  band=%s  day_obs=%s",
                    count,
                    len(refs),
                    visit,
                    band,
                    day_obs,
                )

            # 2. Load source table (NO cache)
            df_src = None
            try:
                df_src = butler.get(ref, parameters={"columns": src_columns_avail})
                if not isinstance(df_src, pd.DataFrame):
                    df_src = df_src.to_pandas()
            except Exception as exc:
                log.warning(
                    "    WARNING: could not load ref (visit=%s det=%s): %s",
                    visit,
                    detector,
                    exc,
                )
                n_failed += 1
                continue

            if df_src is None or len(df_src) == 0:
                n_failed += 1
                del df_src
                continue

            # 3. SkyCoord catalogue for this detector
            ra_arr = df_src[ra_col].values
            dec_arr = df_src[dec_col].values

            # Heuristic: Butler stores coords in radians when max < 2pi + eps.
            unit_sky = u.rad if float(ra_arr.max()) <= 2 * np.pi + 0.1 else u.deg
            cat_sky = SkyCoord(ra=ra_arr * unit_sky, dec=dec_arr * unit_sky)

            # 4. Nearest-neighbour match
            best_i, sep2d, _ = tgt_sky.match_to_catalog_sky(cat_sky)
            sep_arcsec = float(sep2d.to(u.arcsec).value)

            if sep_arcsec > match_radius_arcsec:
                del df_src, ra_arr, dec_arr, cat_sky
                gc.collect()
                continue

            # 5. Extract the single matched row
            n_matched += 1
            matched = df_src.iloc[best_i]

            m_ra = float(matched[ra_col])
            m_dec = float(matched[dec_col])
            if unit_sky == u.rad:
                m_ra = np.degrees(m_ra)
                m_dec = np.degrees(m_dec)

            row: dict = {
                "simbad_id": simbad_id,
                "target_ra": ra_t,
                "target_dec": dec_t,
                "visit": visit,
                "detector": detector,
                "band": band,
                "day_obs": day_obs,
                "physical_filter": physical_filter,
                "sep_arcsec": sep_arcsec,
                "src_ra": m_ra,
                "src_dec": m_dec,
                "sourceId": (
                    int(matched[id_col]) if id_col in matched.index and pd.notna(matched[id_col]) else np.nan
                ),
            }
            for col in photo_cols:
                row[col] = matched.get(col, np.nan)

            batch.append(row)

            del df_src, ra_arr, dec_arr, cat_sky, matched
            gc.collect()

        # Flush this star's matches to disk
        if batch:
            pd.DataFrame(batch).to_csv(out_csv, mode="a", header=False, index=False)
            log.info("  flushed %d rows to disk", len(batch))
        batch.clear()

        log.info(
            "  matched: %d / %d refs  (load failures: %d)",
            n_matched,
            len(refs),
            n_failed,
        )

        summary_rows.append(
            {
                "simbad_id": simbad_id,
                "ra_deg": ra_t,
                "dec_deg": dec_t,
                "n_refs": len(refs),
                "n_matched": n_matched,
                "n_failed": n_failed,
                "status": "ok" if n_matched > 0 else "no_match",
            }
        )

    pd.DataFrame(summary_rows).to_csv(out_summary_csv, index=False)
    log.info("Match summary saved -> %s", out_summary_csv)

###  Plot function

In [ ]:
def make_plots(
    df_lc: pd.DataFrame,
    df_summary: pd.DataFrame,
    dir_figs: str,
    match_radius: float,
) -> None:
    """Generate and save diagnostic plots to *dir_figs*."""
    import matplotlib

    matplotlib.use("Agg")  # non-interactive backend (safe on RSP)
    import matplotlib.pyplot as plt

    os.makedirs(dir_figs, exist_ok=True)

    def savefig(name: str) -> None:
        """Save current figure as PDF and PNG."""
        for ext in ("pdf", "png"):
            plt.savefig(os.path.join(dir_figs, f"{name}.{ext}"), bbox_inches="tight")
        log.info("  -> saved %s.{pdf,png}", name)

    # Separation histogram
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(df_lc["sep_arcsec"], bins=30, edgecolor="k", linewidth=0.5)
    ax.axvline(match_radius, color="red", ls="--", label=f'search radius = {match_radius}"')
    ax.set_xlabel("Separation (arcsec)")
    ax.set_ylabel("Number of matches")
    ax.set_title("Cross-match separation - Simbad targets vs LSST sources (all visits)")
    ax.legend()
    plt.tight_layout()
    savefig("lc_crossmatch_separation_histogram")
    plt.close(fig)

    # Visit count per star/band
    band_counts = df_lc.groupby(["simbad_id", "band"]).size().unstack(fill_value=0)
    fig, ax = plt.subplots(figsize=(max(6, len(band_counts) * 0.5), 4))
    band_counts.plot(kind="bar", ax=ax, width=0.8)
    ax.set_xlabel("Simbad target")
    ax.set_ylabel("Number of matched visits")
    ax.set_title("Visit count per star and band")
    ax.legend(title="band", bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.xticks(rotation=45, ha="right", fontsize=7)
    plt.tight_layout()
    savefig("lc_visits_per_star_band")
    plt.close(fig)

    # psfFlux preview (first 6 stars, all bands)
    bands_to_plot = ["r", "g", "i", "u", "z", "y"]
    band_colors = {
        "u": "purple",
        "g": "blue",
        "r": "green",
        "i": "orange",
        "z": "red",
        "y": "brown",
    }
    flux_col = (
        "psfFlux" if "psfFlux" in df_lc.columns else "calibFlux" if "calibFlux" in df_lc.columns else None
    )

    if flux_col:
        n_stars = min(6, df_lc["simbad_id"].nunique())
        stars = df_lc["simbad_id"].unique()[:n_stars]
        fig, axes = plt.subplots(n_stars, 1, figsize=(10, 3 * n_stars), sharex=False)
        if n_stars == 1:
            axes = [axes]

        for ax, star_id in zip(axes, stars, strict=False):
            df_star = df_lc[df_lc["simbad_id"] == star_id].sort_values(["band", "visit"])
            for band in bands_to_plot:
                df_b = df_star[df_star["band"] == band]
                if len(df_b) == 0:
                    continue
                x = np.arange(len(df_b))
                y = df_b[flux_col].values
                yerr = (
                    df_b[flux_col + "Err"].values if flux_col + "Err" in df_b.columns else np.zeros(len(df_b))
                )
                ax.errorbar(
                    x,
                    y,
                    yerr=yerr,
                    fmt="o",
                    ms=3,
                    lw=0.8,
                    color=band_colors.get(band, "gray"),
                    label=f"{band} ({len(df_b)} pts)",
                )
            ax.set_title(star_id, fontsize=8)
            ax.set_xlabel("Visit index (proxy - MJD to be added)")
            ax.set_ylabel(f"{flux_col} [nJy]")
            ax.legend(fontsize=7, ncol=3)

        plt.suptitle(f"Light curves - {flux_col} (all bands)", y=1.01)
        plt.tight_layout()
        savefig("lc_psfFlux_preview")
        plt.close(fig)

## Logging

- le logging ne marche pas sous jupyter notebook

In [ ]:
# 1. Récupérer le logger racine (ou créez un logger spécifique: logging.getLogger('mon_code'))
log = logging.getLogger()

# 2. Définir le niveau de log global (DEBUG, INFO, WARNING, ERROR)
log.setLevel(logging.INFO)

# 3. Éviter la duplication des handlers si la cellule est exécutée plusieurs fois
if not log.handlers:
    # 4. Créer un handler qui écrit vers la sortie standard (capturée par Jupyter)
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)

    # 5. Définir le format des messages (heure, niveau, nom du logger, message)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)

    # 6. Ajouter le handler au logger
    log.addHandler(handler)

# Petit test pour vérifier que ça fonctionne
log.info("Le logging est configuré et fonctionne dans le notebook !")

## Configuration

**Edit only this cell** to point to the right Butler repository, collections,
input CSV, search radius, and output band.


In [ ]:
# ── Output directories ───────────────────────────────────────────────────
NB_TAG = "MATCHSRC_01"
DIR_DATA_IN = "data_DEEPCCUTOUTS_01_in"  # reuse the same input directory as NB01
DIR_DATA_OUT = f"data_{NB_TAG}_out"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA_OUT, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
log.info(f"Data input : {os.path.abspath(DIR_DATA_IN)}")
log.info(f"Data output: {os.path.abspath(DIR_DATA_OUT)}")
log.info(f"Figs       : {os.path.abspath(DIR_FIGS)}")


# Output paths
out_lc_csv = os.path.join(DIR_DATA_OUT, "all_stars_lightcurves.csv")
out_sum_csv = os.path.join(DIR_DATA_OUT, "lightcurve_match_summary.csv")
dir_per_star = os.path.join(DIR_DATA_OUT, "per_star")

# ── Matplotlib style ────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure to PDF and PNG inside DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


log.info("Configuration done.")

In [ ]:
# ── Butler ────────────────────────────────────────────────────────────────
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

instrument = "LSSTCam"
skymapName = "lsst_cells_v2"

BANDSEL = "r"

# ── Input targets ─────────────────────────────────────────────────────
target_file = "summary_visit_counts_per_star_V17-21_r2.0deg.csv"

# ── Cross-match search radius ─────────────────────────────────────────────
MATCH_RADIUS_ARCSEC = 1.0  # maximum separation for a valid match [arcsec]

DATE_START = "2025-04-01T00:00:00"
DATE_STOP = "2026-07-01T00:00:00"
time_start = Time(DATE_START, format="isot", scale="utc")
time_stop = Time(DATE_STOP, format="isot", scale="utc")
MJD_START = time_start.mjd
MJD_STOP = time_stop.mjd
DELTAMJD_DAYS = MJD_STOP - MJD_START
log.debug(f"MJD ::: start = {MJD_START} , stop = {MJD_STOP} , delta t = {DELTAMJD_DAYS} days")

SRC_COLUMNS = [
    "coord_ra",
    "coord_dec",
    "parentSourceId",
    "x",
    "y",
    "xErr",
    "yErr",
    "ra",
    "dec",
    "raErr",
    "decErr",
    "calibFlux",
    "calibFluxErr",
    # "ap03Flux",
    # "ap03FluxErr",
    # "ap03Flux_flag",
    # "ap06Flux",
    # "ap06FluxErr",
    # "ap06Flux_flag",
    "ap09Flux",
    "ap09FluxErr",
    "ap09Flux_flag",
    "ap12Flux",
    "ap12FluxErr",
    "ap12Flux_flag",
    "ap17Flux",
    "ap17FluxErr",
    "ap17Flux_flag",
    "ap25Flux",
    "ap25FluxErr",
    "ap25Flux_flag",
    "ap35Flux",
    "ap35FluxErr",
    "ap35Flux_flag",
    # "ap50Flux",
    # "ap50FluxErr",
    # "ap50Flux_flag",
    # "ap70Flux",
    # "ap70FluxErr",
    # "ap70Flux_flag",
    "sky",
    "skyErr",
    "psfFlux",
    "psfFluxErr",
    #'ixx','iyy','ixy','ixxPSF','iyyPSF','ixyPSF','ixxDebiasedPSF','iyyDebiasedPSF','ixyDebiasedPSF',
    #'gaussianFlux','gaussianFluxErr',
    "extendedness",
    "sizeExtendedness",
    #'blendedness_abs','blendedness_flag','blendedness_flag_noCentroid','blendedness_flag_noShape',
    "apFlux_12_0_flag",
    # "apFlux_12_0_flag_apertureTruncated",
    # "apFlux_12_0_flag_sincCoeffsTruncated",
    "apFlux_12_0_instFlux",
    "apFlux_12_0_instFluxErr",
    "apFlux_17_0_flag",
    "apFlux_17_0_instFlux",
    "apFlux_17_0_instFluxErr",
    "apFlux_35_0_flag",
    "apFlux_35_0_instFlux",
    "apFlux_35_0_instFluxErr",
    # "apFlux_50_0_flag",
    # "apFlux_50_0_instFlux",
    # "apFlux_50_0_instFluxErr",
    #'normCompTophatFlux_flag','normCompTophatFlux_instFlux','normCompTophatFlux_instFluxErr',
    "extendedness_flag",
    "sizeExtendedness_flag",
    #'footprintArea_value','invalidPsfFlag','jacobian_flag','jacobian_value',
    "localBackground_instFlux",
    "localBackground_instFluxErr",
    "localBackground_flag",
    # "localBackground_flag_noGoodPixels",
    # "localBackground_flag_noPsf",
    #'pixelFlags_bad','pixelFlags_cr','pixelFlags_crCenter','pixelFlags_edge','pixelFlags_interpolated','pixelFlags_interpolatedCenter','pixelFlags_nodata','pixelFlags_offimage','pixelFlags_saturated','pixelFlags_saturatedCenter','pixelFlags_suspect','pixelFlags_suspectCenter',
    #'psfFlux_apCorr','psfFlux_apCorrErr',
    #'psfFlux_area','psfFlux_flag','psfFlux_flag_apCorr','psfFlux_flag_edge','psfFlux_flag_noGoodPixels',
    #'gaussianFlux_flag','centroid_flag','centroid_flag_almostNoSecondDerivative','centroid_flag_badError','centroid_flag_edge','centroid_flag_noSecondDerivative','centroid_flag_notAtMaximum','centroid_flag_resetToPeak',
    #'variance_flag','variance_flag_emptyFootprint','variance_value',
    #'calib_astrometry_used','calib_photometry_reserved','calib_photometry_used','calib_psf_candidate','calib_psf_reserved','calib_psf_used',
    #'deblend_deblendedAsPsf','deblend_hasStrayFlux','deblend_masked','deblend_nChild','deblend_parentTooBig','deblend_patchedTemplate','deblend_rampedTemplate','deblend_skipped','deblend_tooManyPeaks',
    #'hsmPsfMoments_flag','hsmPsfMoments_flag_no_pixels','hsmPsfMoments_flag_not_contained','hsmPsfMoments_flag_parent_source',
    #'iDebiasedPSF_flag','iDebiasedPSF_flag_no_pixels','iDebiasedPSF_flag_not_contained','iDebiasedPSF_flag_parent_source','iDebiasedPSF_flag_galsim','iDebiasedPSF_flag_edge',
    #'hsmShapeRegauss_flag','hsmShapeRegauss_flag_galsim','hsmShapeRegauss_flag_no_pixels','hsmShapeRegauss_flag_not_contained','hsmShapeRegauss_flag_parent_source',
    "sky_source",
    "visit",
    "detector",
    "band",
    "physical_filter",
    "sourceId",
]

log.info("Butler configuration done.")

## Load targets

In [ ]:
target_path = os.path.join(DIR_DATA_IN, target_file)
df_targets = pd.read_csv(target_path)
display(df_targets.head())
log.info("Loaded %d targets from %s", len(df_targets), target_path)

In [ ]:
log.info("Loaded %d targets from %s", len(df_targets), target_path)

## Initialise the Butler

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
log.info(f"Butler initialised | repo: {repo}")

## Auto-discover the object-table dataset type

The catalogue dataset name changed across pipeline versions:

| Pipeline era | Dataset type name          |
|:-------------|:---------------------------|
| Gen2 / HSC   | `deepCoadd_obj`            |
| DP1          | `objectTable_tract`        |
| DP2+         | `object_table_tract`       |

We probe the registry so the notebook is collection-agnostic.


In [ ]:
# Prioritised list of candidate object-table dataset type names
SRC_TABLE_CANDIDATES = [
    "source",
    "sourceTable," "sourceTable_visit",  # visit-level source table (fallback)
]

# List all source-table-related types actually in the registry
all_src_types = [
    d.name for d in registry.queryDatasetTypes() if "source" in d.name.lower() or "table" in d.name.lower()
]
# print("src-table-related dataset types in registry:")
# for t in sorted(all_src_types):
#    print(f"  {t}")

# Pick the first candidate that is registered
SRC_DATASET = None
for name in SRC_TABLE_CANDIDATES:
    if dataset_type_exists(butler, name):
        SRC_DATASET = name
        log.info(f"\n✔ Selected src-table dataset type: '{SRC_DATASET}'")
        break

if SRC_DATASET is None:
    raise RuntimeError(
        "No recognised source-table dataset type found in this Butler collection. "
        f"Candidate types seen: {all_src_types}"
    )

## Inspect the schema of the source table (probe on first target)

We load a single source table from a representative visit to check which
columns are available before the main loop.

In [ ]:
# Use the coordinates of the first target to locate a representative tract
first_row = df_targets.iloc[0]

target_name = first_row["simbad_id"]
target_field = first_row["field"]
target_ra = first_row["ra_deg"]
target_dec = first_row["dec_deg"]

probe_point = SpherePoint(target_ra, target_dec, degrees)
probe_tract_info = skymap.findTract(probe_point)
probe_patch_info = probe_tract_info.findPatch(probe_point)
probe_tract_id = probe_tract_info.getId()
probe_patch_id = probe_patch_info.getSequentialIndex()
probe_dataId = {"tract": probe_tract_id, "patch": probe_patch_id, "skymap": skymapName}

# Build timespan
t1 = Time(MJD_START, format="mjd", scale="tai")
t2 = Time(MJD_STOP, format="mjd", scale="tai")
timespan = Timespan(t1, t2)
log.info("Timespan MJD [%.1f, %.1f]  (delta=%.0f days)", t1.mjd, t2.mjd, t2.mjd - t1.mjd)

In [ ]:
ra_col, dec_col, id_col, src_cols_avail = probe_schema(
    butler,
    timespan,
    ra=float(first_row["ra_deg"]),
    dec=float(first_row["dec_deg"]),
    src_columns=SRC_COLUMNS,
)

In [ ]:
log.info(f"\t RA  column : {ra_col}")
log.info(f"\t Dec column : {dec_col}")
log.info(f"\t ID  column : {id_col}")

# Restrict SRC_COLUMNS to those actually present in this schema
src_cols_avail = [c for c in SRC_COLUMNS if c in df_probe.columns]
log.info(f"Requested columns available: {len(src_cols_avail)}/{len(SRC_COLUMNS)}")

### Test reading of sources with butler.query_datasets()

In [ ]:
# Query source refs for the probe star — materialise into a list
probe_refs = list(
    butler.query_datasets(
        "source",
        where=(
            "visit.timespan OVERLAPS :timespan AND " "visit_detector_region.region OVERLAPS POINT(:ra, :dec)"
        ),
        bind={"timespan": timespan, "ra": target_ra, "dec": target_dec},
    )
)

# Number of refs == number of (visit, detector) pairs covering this position
log.info(f"Number of source refs (visits x detectors): {len(probe_refs)}")
if probe_refs:
    msg = probe_refs[0].dataId.to_json()
    log.info(f"Example dataId: {msg}")

# Load the first source table to probe the schema
df_probe = butler.get(probe_refs[0], parameters={"columns": SRC_COLUMNS})
if not isinstance(df_probe, pd.DataFrame):
    df_probe = df_probe.to_pandas()
log.info(f"Probe source table: {len(df_probe)} rows, {len(df_probe.columns)} columns")

In [ ]:
logging.info(f"Columns of sources dataframe : {df_probe.columns.tolist()}")

## Main loop

In [ ]:
extract_lightcurves(
    butler=butler,
    df_targets=df_targets,
    timespan=timespan,
    ra_col=ra_col,
    dec_col=dec_col,
    id_col=id_col,
    src_columns_avail=src_cols_avail,
    match_radius_arcsec=MATCH_RADIUS_ARCSEC,
    out_csv=out_lc_csv,
    out_summary_csv=out_sum_csv,
)

## Read back the full LC table (on disk -> small RAM footprint)

In [ ]:
df_lc = pd.read_csv(out_lc_csv)
df_summary = pd.read_csv(out_sum_csv)
log.info("df_lc: %d rows x %d cols", len(df_lc), len(df_lc.columns))

## Save light curves

- **`all_stars_lightcurves.csv/.parquet`** — full table, all stars, all bands.
- **`per_star/`** — one CSV per star (named by `simbad_id` with special chars replaced).

> MJD timestamps will be joined in a later notebook from the visit table
> (`visitTable` / `consDb`). For now we keep `visit` and `day_obs` as integer keys.


### Parquet copy of the global LC table

In [ ]:
out_lc_parquet = os.path.join(DIR_DATA_OUT, "all_stars_lightcurves.parquet")
df_lc.to_parquet(out_lc_parquet, index=False)
log.info("Parquet copy saved -> %s", out_lc_parquet)

### Per-star files

In [ ]:
# Per-star files
save_per_star(df_lc, dir_per_star)

## Diagnostic plots

In [ ]:
make_plots(df_lc, df_summary, DIR_FIGS, MATCH_RADIUS_ARCSEC)
log.info("Done.")

## End of notebook

The next step is to join `df_lc` with the Rubin `visitTable` (or `consDb`)
to attach precise MJD timestamps to each measurement.
This will be done in notebook `03_AddMJDFromVisitTable.ipynb`.
